In [4]:
import sys
sys.path.append(r"C:\ddddwoo\project\HotelBear")
from BackEnd.database import get_connection
import numpy as np
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import tqdm 
import re

In [5]:
def load_data():
    conn = get_connection()
    with conn.cursor() as cursor:
                cursor.execute("SELECT * FROM onlyhotellist")
                rows = cursor.fetchall()  # DictCursor라서: 리스트[딕셔너리, ...]
    df = pd.DataFrame(rows)      # 딕셔너리 리스트 → DataFrame
    return df

In [2]:
data = {
    'type': [100, 500, 100, 500],
    'area': [101, 102, 103, 111],
    'name': ['서울호텔', '경기리조트', '인천호텔', '제주리조트'],
    'low_week_price': [150000, 250000, 120000, 350000],
    'grade': [4.5, 4.2, 3.8, 4.8],
    'reviewCount': [120, 80, 50, 300],
    'facility_list': ['수영장,조식,와이파이', '수영장,사우나', '주차장', '수영장,조식,바,클럽'],
    'swimming_pool': [2, 3, 0, 1],
    'star': [5, 4, 3, 5],
    'parking': [1, 1, 1, 1]
}

In [6]:
df = load_data()
display(df.head())

,id,type,area,name,low_week_price,low_weekend_price,event_price,grade,reviewCount,facillity_list,...,pickUp,fitness,bar,desk24,terrace,club,parking,address,img_url,star
0,125,100,101,포시즌스 호텔 서울,1089000,1210000,1210000,5.0,132,"실내 수영장,비즈니스,금연객실,욕조,테라스/발코니,와이파이,피트니스 센터,24시간데...",...,0,1,1,1,1,2,1,서울특별시 종로구 새문안로 97,https://yaimg.yanolja.com/v5/2023/01/13/12/128...,5.0
1,126,100,101,콘래드 서울,770000,917400,935000,4.8,476,"실내 수영장,사우나,비즈니스,금연객실,욕조,테라스/발코니,와이파이,피트니스 센터,2...",...,0,1,1,1,1,2,1,서울특별시 영등포구 국제금융로 10(여의도동),https://yaimg.yanolja.com/v5/2024/07/03/13/128...,5.0
2,127,100,101,그랜드 인터컨티넨탈 서울 파르나스,774400,1076900,726000,5.0,838,"실내 수영장,사우나,비즈니스,금연객실,커플룸,욕조,테라스/발코니,와이파이,피트니스 ...",...,0,1,1,1,1,2,1,서울특별시 강남구 테헤란로 521,https://yaimg.yanolja.com/v5/2025/11/18/03/128...,5.0
3,128,100,101,JW 메리어트 호텔 서울,701800,701800,665500,4.9,207,"실내 수영장,사우나,비즈니스,금연객실,욕조,와이파이,피트니스 센터,24시간데스크,수...",...,0,1,1,1,0,2,1,서울특별시 서초구 신반포로 176 (센트럴시티),https://yaimg.yanolja.com/v5/2025/07/09/08/128...,5.0
4,129,100,101,아난티 앳 강남,547520,840160,708000,4.7,87,"실내 수영장,야외수영장,사우나,금연객실,와이파이,피트니스 센터,24시간데스크,어메니...",...,0,1,0,1,0,0,1,서울특별시 강남구 논현로 734,https://yaimg.yanolja.com/v5/2024/08/19/10/128...,4.0


In [7]:
df['facility_count'] = df['facillity_list'].apply(lambda x: len(x.split(','))if isinstance(x, str) else 0)

df = pd.get_dummies(df, columns=['area', 'type'], prefix=['area', 'type'])

id_cols = [col for col in df.columns if col not in ['low_week_price', 'low_weekend_price', 'event_price', 'name', 'facillity_list']]

df_melted = pd.melt(
    df, 
    id_vars=id_cols, 
    value_vars=['low_week_price', 'low_weekend_price', 'event_price'],
    var_name='price_type', 
    value_name='target_price'
)

price_map = {'low_week_price': 0, 'low_weekend_price': 1, 'event_price': 2}
df_melted['day_feature'] = df_melted['price_type'].map(price_map)

df_melted['log_target_price'] = np.log1p(df_melted['target_price'])

final_df = df_melted.drop(columns=['price_type', 'target_price'])

print(f"전처리 전 데이터 개수: {len(df)}개")
print(f"전처리 후 데이터 개수: {len(final_df)}개 (3배 증폭 완료)")
print("\n--- 최종 데이터 컬럼 구성 ---")
print(final_df.columns.tolist())

전처리 전 데이터 개수: 904개
전처리 후 데이터 개수: 2712개 (3배 증폭 완료)

--- 최종 데이터 컬럼 구성 ---
['id', 'grade', 'reviewCount', 'swimming_pool', 'breakfast', 'bathtub', 'pickUp', 'fitness', 'bar', 'desk24', 'terrace', 'club', 'parking', 'address', 'img_url', 'star', 'facility_count', 'area_101', 'area_102', 'area_103', 'area_104', 'area_105', 'area_106', 'area_107', 'area_108', 'area_109', 'area_110', 'area_111', 'type_100', 'type_500', 'day_feature', 'log_target_price']


In [8]:
geolocator = Nominatim(user_agent="hotel_price_predictor", timeout=10)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=2)

def clean_address(address):
    return re.sub(r'\([^)]*\)', '', address).strip()

# 2. 좌표를 가져오는 함수 정의
def get_coordinates(address):
    # 1단계: 주소 정제 (괄호 제거)
    cleaned_addr = clean_address(address)
    
    try:
        location = geocode(cleaned_addr)
        if location:
            return location.latitude, location.longitude
        
        short_addr = " ".join(cleaned_addr.split()[:3])
        location = geocode(short_addr)
        if location:
            return location.latitude, location.longitude
            
        return None, None
    except Exception as e:
        print(f"⚠️ 에러 발생: {e}")
        return None, None

print("📍 주소를 좌표로 변환 중입니다... (잠시만 기다려주세요)")

# 중복된 주소만 따로 뽑아서 변환하면 시간을 훨씬 아낄 수 있습니다.
unique_addresses = final_df['address'].unique()
address_map = {}
for addr in unique_addresses:
    lat, lon = get_coordinates(addr)
    address_map[addr] = {'lat': lat, 'lon': lon}


final_df['latitude'] = final_df['address'].map(lambda x: address_map[x]['lat'])
final_df['longitude'] = final_df['address'].map(lambda x: address_map[x]['lon'])

final_df = final_df.dropna(subset=['latitude', 'longitude'])

print(f"✅ 변환 완료! 현재 데이터 개수: {len(final_df)}개")

📍 주소를 좌표로 변환 중입니다... (잠시만 기다려주세요)


KeyboardInterrupt: 

In [ ]:
from haversine import haversine
import matplotlib.pyplot as plt
import seaborn as sns

center_loc = (33.4996, 126.5312) 

def calculate_distance(row):
    hotel_loc = (row['latitude'], row['longitude'])
    # 단위: km (미터로 하려면 * 1000)
    return haversine(center_loc, hotel_loc)

# 모든 호텔 행에 대해 중심지와의 거리 계산
final_df['dist_to_center'] = final_df.apply(calculate_distance, axis=1)

# 1-2. 데이터 정제 (학습에 방해되는 텍스트 컬럼 제거)
# id, address, img_url 등은 숫자가 아니므로 모델이 읽지 못합니다.
features_to_drop = ['id', 'address', 'img_url', 'latitude', 'longitude'] # 좌표 자체보다 '거리'가 더 좋은 힌트가 될 때가 많음
study_df = final_df.drop(columns=features_to_drop)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 2-1. 데이터 분리 (문제와 정답)
X = study_df.drop(columns=['log_target_price']) # 문제집
y = study_df['log_target_price'] # 정답지

# 학습 데이터와 테스트 데이터 분리 (8:2 비율)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2-2. 3가지 강력한 모델 정의
model1 = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=7, random_state=42)
model2 = LGBMRegressor(n_estimators=1000, learning_rate=0.05, num_leaves=31, random_state=42)
model3 = RandomForestRegressor(n_estimators=500, max_depth=10, random_state=42)

# 2-3. 앙상블 모델 구성 (Voting)
ensemble_model = VotingRegressor(
    estimators=[
        ('xgb', model1), 
        ('lgbm', model2), 
        ('rf', model3)
    ]
)

# 2-4. 학습 시작
print("🚀 앙상블 모델 학습 중...")
ensemble_model.fit(X_train, y_train)

# 2-5. 평가
y_pred_log = ensemble_model.predict(X_test)
# 로그를 다시 원래 가격으로 복구 (np.expm1)
y_test_real = np.expm1(y_test)
y_pred_real = np.expm1(y_pred_log)

mae = mean_absolute_error(y_test_real, y_pred_real)
r2 = r2_score(y_test, y_pred_log)

print(f"\n📊 모델 평가 결과")
print(f"평균 절대 오차(MAE): 약 {int(mae):,}원")
print(f"결정계수(R2 Score): {r2:.4f} (1에 가까울수록 완벽)")

In [ ]:
# 1. 한글 폰트 설정 (Windows 기준, Mac은 'AppleGothic' 사용)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 그래프 크기 설정
plt.figure(figsize=(18, 5))

# --- [그래프 1] 실제값 vs 예측값 (Scatter Plot) ---
# 모델이 얼마나 정답에 가깝게 예측했는지 점을 찍어보는 거예요.
plt.subplot(1, 3, 1)
sns.scatterplot(x=y_test_real, y=y_pred_real, alpha=0.5)
# 완벽한 정답선(대각선) 그리기
plt.plot([y_test_real.min(), y_test_real.max()], [y_test_real.min(), y_test_real.max()], 'r--', lw=2)
plt.xlabel('실제 가격 (원)')
plt.ylabel('예측 가격 (원)')
plt.title('실제값 vs 예측값 분포')

# --- [그래프 2] 오차 분포 (Residual Plot) ---
# 모델이 주로 얼마만큼의 오차를 내는지 확인합니다. 0 근처에 몰려있을수록 좋아요.
plt.subplot(1, 3, 2)
error = y_test_real - y_pred_real
sns.histplot(error, kde=True, color='skyblue')
plt.axvline(x=0, color='red', linestyle='--')
plt.xlabel('오차 (실제값 - 예측값)')
plt.title('예측 오차 분포 (Residuals)')

# --- [그래프 3] 변수 중요도 (Feature Importance) ---
# 앙상블 모델 중 XGBoost 모델을 기준으로 "어떤 데이터가 가격에 가장 큰 영향을 줬는지" 확인합니다.
plt.subplot(1, 3, 3)
# 앙상블 내의 첫 번째 모델(XGBoost)의 중요도 추출
importances = ensemble_model.estimators_[0].feature_importances_
feature_names = X.columns
indices = np.argsort(importances)[-15:] # 상위 15개만

plt.barh(range(len(indices)), importances[indices], align='center')
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel('중요도')
plt.title('가격 결정 핵심 요소 Top 15 (XGBoost 기준)')

plt.tight_layout()
plt.show()